# 3.7 softmax 回归的简洁实现

3.6 节手写了 softmax 回归的各个组件。本节使用 PyTorch 高级 API 实现同一个模型，并重点理解框架如何通过“logits + 交叉熵”保证数值稳定。

原教材：[3.7 softmax 回归的简洁实现](https://zh.d2l.ai/chapter_linear-networks/softmax-regression-concise.html)

## 学习目标

1. 使用 `nn.Flatten` 和 `nn.Linear` 定义 softmax 回归；
2. 使用 `apply` 与 `nn.init.normal_` 初始化权重；
3. 理解为什么模型应输出 logits，而不是预先输出 softmax 概率；
4. 掌握 `CrossEntropyLoss`、`SGD` 和标准训练循环；
5. 分析学习率、批量大小、迭代周期和过拟合。

In [ ]:
import sys  # 用于判断操作系统
from pathlib import Path  # 用于表示数据目录

import matplotlib.pyplot as plt  # 用于绘制训练曲线
import torch  # 导入 PyTorch
import torchvision  # 提供 Fashion-MNIST
from torch import nn  # 导入神经网络模块
from torch.utils import data  # 导入 DataLoader
from torchvision import transforms  # 导入图像变换

torch.manual_seed(42)  # 固定随机种子

def load_data_fashion_mnist(batch_size):  # 定义独立的数据加载函数
    """下载 Fashion-MNIST，并返回训练与测试迭代器。"""
    transform = transforms.ToTensor()  # 图像转为 [0,1] 浮点张量
    root = Path("../data")  # 数据缓存目录
    train_set = torchvision.datasets.FashionMNIST(  # 创建训练集
        root=root, train=True, transform=transform, download=True
    )
    test_set = torchvision.datasets.FashionMNIST(  # 创建测试集
        root=root, train=False, transform=transform, download=True
    )
    workers = 0 if sys.platform.startswith("win") else 4  # Windows Notebook 避免多进程问题
    train_loader = data.DataLoader(  # 创建训练加载器
        train_set, batch_size=batch_size, shuffle=True, num_workers=workers
    )
    test_loader = data.DataLoader(  # 创建测试加载器
        test_set, batch_size=batch_size, shuffle=False, num_workers=workers
    )
    return train_loader, test_loader  # 返回两个加载器

batch_size = 256  # 与原教材一致
train_iter, test_iter = load_data_fashion_mnist(batch_size)  # 首次运行会下载数据

## 3.7.1 初始化模型参数

softmax 回归只有一个全连接输出层。PyTorch 不会自动把 `[B,1,28,28]` 图像展平，因此先使用 `nn.Flatten()` 得到 `[B,784]`，再用 `nn.Linear(784,10)` 得到 10 个 logits：

$$
[B,1,28,28]\rightarrow[B,784]\rightarrow[B,10].
$$

`nn.Sequential` 是按顺序调用各层的容器。这里只有两层操作，但这种写法可以自然扩展到深层网络。

In [ ]:
net = nn.Sequential(  # 创建按顺序执行的网络
    nn.Flatten(),  # 保留批量维，把其余维度展平：[B,1,28,28] -> [B,784]
    nn.Linear(28 * 28, 10),  # 线性变换：[B,784] -> [B,10]
)

def init_weights(module):  # 定义供 net.apply 调用的初始化函数
    if isinstance(module, nn.Linear):  # 只处理线性层，不处理 Flatten
        nn.init.normal_(module.weight, mean=0.0, std=0.01)  # 原地正态初始化权重
        nn.init.zeros_(module.bias)  # 将偏置初始化为 0

net.apply(init_weights)  # 递归访问网络中的每个模块并调用 init_weights
print(net)  # 查看网络结构
print("权重形状:", net[1].weight.shape)  # PyTorch Linear 权重形状为 [输出数,输入数]=[10,784]
print("偏置形状:", net[1].bias.shape)  # [10]

### 必要的 Python 与 PyTorch 语法

- `net[1]` 取 `Sequential` 中索引为 1 的第二个模块；
- `isinstance(module, nn.Linear)` 判断对象是否为线性层；
- 函数名末尾的 `_`，如 `normal_`、`zeros_`，通常表示原地修改张量；
- `net.apply(function)` 会递归遍历所有子模块，把每个模块传给该函数；
- PyTorch 的 `Linear` 保存权重为 `[out_features, in_features]`，前向传播内部等价于 $XW^\top+b$。

In [ ]:
shape_test_images = torch.rand(4, 1, 28, 28)  # 构造 4 张合成图像
shape_test_logits = net(shape_test_images)  # 执行完整前向传播
print("输入形状:", shape_test_images.shape)  # [4,1,28,28]
print("logits 形状:", shape_test_logits.shape)  # [4,10]
print("第一条 logits:", shape_test_logits[0])  # logits 不要求为正，也不要求和为 1

## 3.7.2 重新审视 softmax 的实现

按定义计算

$$
\hat y_j=\frac{\exp(o_j)}{\sum_k\exp(o_k)}
$$

可能产生两个问题：大正数求指数会**上溢**为 `inf`；大负数求指数会**下溢**为 0，随后 `log(0)` 得到 `-inf`。

softmax 对整体平移不敏感，可以先令 $m=\max_k o_k$：

$$
\operatorname{softmax}(\mathbf o)_j
=\frac{\exp(o_j-m)}{\sum_k\exp(o_k-m)}.
$$

但若先得到概率再取对数，极小概率仍可能舍入为 0。更稳定的做法是把 softmax 与交叉熵合并：

$$
-\log\hat y_y
=\log\sum_k\exp(o_k)-o_y.
$$

因此模型输出原始 logits，`CrossEntropyLoss` 内部以稳定方式完成 `log_softmax + NLLLoss`。训练前不要再手动添加 `Softmax`。

In [ ]:
extreme_logits = torch.tensor([[1000.0, 999.0, -1000.0]])  # 构造极端 logits
extreme_label = torch.tensor([0])  # 真实类别为第 0 类

naive_exp = torch.exp(extreme_logits)  # 直接求指数会出现 inf
naive_probability = naive_exp / naive_exp.sum(dim=1, keepdim=True)  # inf/inf 产生 nan
stable_probability = torch.softmax(extreme_logits, dim=1)  # 框架 softmax 使用稳定实现
stable_loss = nn.functional.cross_entropy(extreme_logits, extreme_label)  # 直接从 logits 算损失

print("朴素概率:", naive_probability)
print("稳定概率:", stable_probability)
print("稳定交叉熵:", stable_loss.item())

In [ ]:
loss = nn.CrossEntropyLoss(reduction="none")  # 返回每个样本的损失，形状为 [B]

example_logits = torch.tensor([[2.0, 1.0, 0.1],  # 两个样本的未归一化分数
                               [0.2, 0.4, 1.5]])
example_labels = torch.tensor([0, 2])  # 真实类别索引，类型必须是 torch.int64
print("逐样本损失:", loss(example_logits, example_labels))  # 损失函数直接接收 logits
print("预测概率:", torch.softmax(example_logits, dim=1))  # 只有展示概率时才显式 softmax

## 3.7.3 优化算法

`torch.optim.SGD` 接收网络参数迭代器并管理更新。`net.parameters()` 会依次提供可训练的权重和偏置。优化器不计算梯度；梯度由 `backward()` 计算，优化器只根据已有梯度执行 `step()`。

In [ ]:
learning_rate = 0.1  # 与原教材一致
trainer = torch.optim.SGD(  # 创建小批量随机梯度下降优化器
    net.parameters(),  # 把 net 中所有可训练参数交给优化器管理
    lr=learning_rate,  # 每次沿负梯度方向移动的步长
)
print(trainer)  # 查看优化器配置

## 3.7.4 训练

标准 PyTorch 训练顺序为：

```python
trainer.zero_grad()   # 清梯度
logits = net(X)       # 前向传播
batch_loss = loss(logits, y).mean()
batch_loss.backward() # 计算梯度
trainer.step()        # 更新参数
```

必须清梯度，因为 PyTorch 默认把新梯度累加到已有 `.grad` 中。

In [ ]:
def number_correct(logits, labels):  # 计算一个批量中的正确预测数
    predictions = logits.argmax(dim=1)  # 最大 logit 的索引就是预测类别
    return int((predictions == labels).sum())  # 布尔值求和后转为 Python 整数

def evaluate_accuracy(net, data_iter):  # 计算完整数据集准确率
    net.eval()  # 切换到评估模式；对未来的 Dropout、BatchNorm 很重要
    correct = 0  # 累计正确预测数
    total = 0  # 累计样本数
    with torch.no_grad():  # 评估阶段关闭梯度记录
        for X, y in data_iter:  # 遍历所有批量
            logits = net(X)  # 得到未归一化输出
            correct += number_correct(logits, y)  # 累计正确数
            total += y.numel()  # 累计标签数量
    return correct / total  # 返回准确率

In [ ]:
def train_model(net, train_iter, test_iter, loss, trainer, num_epochs):  # 完整训练函数
    history = {"train_loss": [], "train_acc": [], "test_acc": []}  # 保存训练历史

    for epoch in range(num_epochs):  # 重复遍历训练集
        net.train()  # 切换到训练模式
        loss_sum = 0.0  # 累计逐样本损失总和
        correct = 0  # 累计正确预测数
        total = 0  # 累计样本数

        for X, y in train_iter:  # 逐批训练
            trainer.zero_grad()  # 清除上一个批量留下的梯度
            logits = net(X)  # 前向传播；不要手动 softmax
            losses = loss(logits, y)  # 得到当前批量的逐样本损失
            losses.mean().backward()  # 对平均损失反向传播
            trainer.step()  # 根据梯度更新权重和偏置

            loss_sum += losses.sum().item()  # 累计损失总和
            correct += number_correct(logits, y)  # 累计正确数
            total += y.numel()  # 累计样本数

        train_loss = loss_sum / total  # 计算每个样本的平均训练损失
        train_acc = correct / total  # 计算训练准确率
        test_acc = evaluate_accuracy(net, test_iter)  # 计算测试准确率
        history["train_loss"].append(train_loss)  # 保存指标
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)
        print(  # 显示本轮结果
            f"epoch {epoch + 1:2d}: loss={train_loss:.4f}, "
            f"train acc={train_acc:.4f}, test acc={test_acc:.4f}"
        )

    return history  # 返回各轮指标

In [ ]:
num_epochs = 10  # 与原教材一致，训练 10 轮
history = train_model(net, train_iter, test_iter, loss, trainer, num_epochs)  # 开始训练

epochs = range(1, num_epochs + 1)  # 创建横轴
plt.plot(epochs, history["train_loss"], label="train loss")  # 绘制训练损失
plt.plot(epochs, history["train_acc"], label="train acc")  # 绘制训练准确率
plt.plot(epochs, history["test_acc"], label="test acc")  # 绘制测试准确率
plt.xlabel("epoch")  # 设置横轴标题
plt.grid(alpha=0.3)  # 显示浅色网格
plt.legend()  # 显示图例
plt.show()  # 显示图像

## 本节知识与代码梳理

| 从零实现 | 简洁实现 | 作用 |
|---|---|---|
| `reshape` | `nn.Flatten` | 展平图像 |
| `W`、`b` | `nn.Linear(784,10)` | 保存并计算线性参数 |
| 手写 softmax 与负对数 | `nn.CrossEntropyLoss` | 稳定计算分类损失 |
| 手写 `sgd` | `torch.optim.SGD` | 更新参数 |
| 手工参数列表 | `net.parameters()` | 遍历可训练参数 |

关键数据流：

$$
X\xrightarrow{\text{Flatten}}X_{flat}
\xrightarrow{\text{Linear}}\text{logits}
\xrightarrow{\text{CrossEntropyLoss}}\text{loss}.
$$

训练时损失接收 logits；需要展示概率时才调用 `torch.softmax(logits, dim=1)`。

## 3.7.5 小结

- 高级 API 能显著减少模型、损失和优化器的样板代码；
- `nn.Linear` 的输出是 logits，不是概率；
- softmax 与交叉熵应合并计算，以避免上溢、下溢和 `log(0)`；
- `CrossEntropyLoss` 直接接收 logits 和整数类别标签；
- 框架实现不仅更短，通常还包含经过测试的数值稳定与性能优化。

## 3.7.6 练习

1. 调整批量大小、迭代周期数和学习率，并观察结果。
2. 增加迭代周期数。为什么测试精度可能在一段时间后降低？如何解决？

## 3.7.7 练习题参考答案

### 练习 1

三个超参数的典型影响如下：

- **学习率过小**：损失下降慢，相同轮数下可能尚未收敛；
- **学习率过大**：损失震荡、发散，甚至出现非有限数值；
- **批量较小**：每轮更新次数多、梯度噪声大，可能改善泛化，但硬件吞吐量较低；
- **批量较大**：梯度更稳定、矩阵运算效率高，但占用更多内存，每轮更新次数减少；
- **迭代周期太少**：欠拟合；增加周期通常先改善结果，之后收益减小并可能过拟合。

实验时一次只改变一个变量，并记录训练损失、训练准确率、验证准确率和运行时间。不能根据测试集反复选择超参数，应从训练数据中划出验证集。一个合理的起点是：`batch_size` 取 64、128、256；学习率取 0.01、0.03、0.1、0.3；训练 5～20 轮。具体最佳值依赖模型、数据和硬件。

In [ ]:
def build_softmax_model():  # 每次实验都创建全新模型，保证公平比较
    model = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 10))  # 定义模型
    model.apply(init_weights)  # 使用相同规则初始化
    return model  # 返回新模型

def run_experiment(batch_size=256, learning_rate=0.1, num_epochs=10):  # 实验模板
    experiment_train_iter, experiment_test_iter = load_data_fashion_mnist(batch_size)  # 加载数据
    model = build_softmax_model()  # 创建独立模型
    criterion = nn.CrossEntropyLoss(reduction="none")  # 创建损失函数
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)  # 创建优化器
    result = train_model(  # 按指定配置训练
        model, experiment_train_iter, experiment_test_iter,
        criterion, optimizer, num_epochs,
    )
    return model, result  # 返回模型和训练历史

# 示例：取消下一行注释即可进行一组实验
# experiment_model, experiment_history = run_experiment(128, 0.1, 10)

### 练习 2

训练轮数增加后，模型可能越来越贴合训练集中的偶然模式和噪声，而不是可推广规律。这叫**过拟合**。典型现象是：训练损失继续下降、训练准确率继续上升，但验证或测试准确率开始下降。

解决方法包括：

1. 划分验证集并使用**早停**：保存验证指标最佳的模型，而不是最后一轮模型；
2. 使用权重衰减等正则化，限制参数过度增大；
3. 收集更多训练数据或使用合理的数据增强；
4. 减小模型容量；
5. 使用学习率衰减，使训练后期更新更细致；
6. 检查训练集与测试集是否同分布，以及标签是否存在噪声。

测试集只应在模型和超参数确定后用于最终报告。若一边观察测试精度一边决定停止轮数，测试集事实上已经参与了模型选择，最终结果会偏乐观。